# Lift Measure in Machine Learning - Starter Notebook
Computes and visualizes the Lift metric for association rules: Lift > 1 indicates positive correlation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

In [ ]:
transactions = [
    {'bread','milk','butter'},
    {'bread','butter'},
    {'milk','bread'},
    {'bread','milk','butter','eggs'},
    {'bread','eggs'},
    {'milk','butter'},
    {'milk','eggs'},
    {'bread','butter','eggs'},
    {'milk','bread','eggs'},
    {'butter','eggs'},
    {'bread','milk'},
    {'bread','butter','milk'},
]
N = len(transactions)
print(f'{N} transactions loaded')

In [ ]:
def sup(itemset):
    fs = frozenset(itemset)
    return sum(1 for t in transactions if fs <= t) / N

items = sorted({i for t in transactions for i in t})

# Compute lift for all 2-item rules
records = []
for a, b in combinations(items, 2):
    sup_ab = sup([a, b])
    sup_a  = sup([a])
    sup_b  = sup([b])
    if sup_a > 0 and sup_b > 0:
        lift_ab = sup_ab / (sup_a * sup_b)
        conf_ab = sup_ab / sup_a
        conf_ba = sup_ab / sup_b
        records.append({'A': a, 'B': b,
                        'sup(A)': round(sup_a,3),
                        'sup(B)': round(sup_b,3),
                        'sup(A,B)': round(sup_ab,3),
                        'conf(A->B)': round(conf_ab,3),
                        'conf(B->A)': round(conf_ba,3),
                        'lift': round(lift_ab,3)})

df = pd.DataFrame(records).sort_values('lift', ascending=False)
print(df.to_string(index=False))

In [ ]:
# Lift interpretation
print('\nInterpretation:')
for _, row in df.iterrows():
    if row['lift'] > 1:
        tag = 'positive association'
    elif row['lift'] < 1:
        tag = 'negative association'
    else:
        tag = 'independent'
    print(f"  {row['A']} & {row['B']}: lift={row['lift']:.3f} -> {tag}")

In [ ]:
# Bar chart of lift values
labels = [f"{r['A']},{r['B']}" for _, r in df.iterrows()]
lifts  = df['lift'].values
colors = ['#2ecc71' if l > 1 else '#e74c3c' for l in lifts]

plt.figure(figsize=(8, 4))
plt.bar(labels, lifts, color=colors, edgecolor='k')
plt.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Lift = 1 (independent)')
plt.ylabel('Lift')
plt.xlabel('Item pair')
plt.title('Lift Values for 2-item Pairs')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.show()